# YOLOv8n 학습 -- YOLOv8s/YOLO26n과 동일 조건 3-way 비교용

`train_yolo_tile_augmented.ipynb`가 이미 만들어둔 증강 데이터셋
(`rummikub_data_3_tile_augmented`, 7:2:1 train/val/test, seed=42)을 그대로 재사용해서
**YOLOv8n만** 추가로 학습합니다. 데이터셋을 다시 만들지 않습니다 -- 같은 `OUT_ROOT`
경로에 이미 있어야 하고, 없으면 이 노트북은 바로 실패합니다(아래 셀 참고).

이 노트북의 목적은 `EXPERIMENT_PLAN.md` 4.2절의 2차(YOLOv8n 3-way 확장)입니다:
YOLOv8n이 YOLO26n과 동일한 train/val/test split, 이미지 크기, epoch, batch, augmentation,
seed, checkpoint 선택 기준으로 학습됐다는 걸 보장해야 3-way 비교가 의미 있습니다. 그래서
학습 하이퍼파라미터(`YOLO_EPOCHS`, `YOLO_IMGSZ`, `batch=-1`, `seed`, `patience`,
`close_mosaic` 등)는 `train_yolo_tile_augmented.ipynb`의 YOLOv8s/YOLO26n 학습 셀과
값을 전부 동일하게 맞췄습니다.

cwd가 `.../boardgame-ai/rummikub_0730`라고 가정한 상대경로만 쓰며, 아직 실행하지 않은
상태입니다.

In [ ]:
from __future__ import annotations

import random
import shutil
from pathlib import Path

import torch

# train_yolo_tile_augmented.ipynb와 동일한 경로/시드 -- 새 데이터셋을 만들지 않고
# 그 노트북이 이미 만들어둔 결과물을 그대로 가리키기만 한다.
OUT_ROOT = Path("rummikub_data_3_tile_augmented")
WEIGHTS_OUT = Path("weights")
WEIGHTS_OUT.mkdir(exist_ok=True)

RANDOM_SEED = 42
random.seed(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

## 1. 데이터셋 재사용 확인

`train_yolo_tile_augmented.ipynb`를 먼저 실행해서 `dataset.yaml`이 이미 만들어져 있어야
한다. 없으면 여기서 바로 에러를 내고 멈춘다 -- 조용히 다른 데이터로 새로 만들면 YOLOv8s/
YOLO26n과 다른 데이터로 학습하게 돼서 3-way 비교 자체가 무효가 된다.

In [ ]:
dataset_yaml_path = OUT_ROOT / "dataset.yaml"
if not dataset_yaml_path.is_file():
    raise FileNotFoundError(
        f"{dataset_yaml_path.resolve()}가 없습니다. "
        "train_yolo_tile_augmented.ipynb의 1~6절(데이터셋 생성)을 먼저 실행하세요 -- "
        "여기서 데이터셋을 새로 만들면 YOLOv8s/YOLO26n과 다른 데이터가 되어 "
        "3-way 비교가 무효가 됩니다."
    )
print(dataset_yaml_path.read_text(encoding="utf-8"))

for split in ("train", "val", "test"):
    n_images = len(list((OUT_ROOT / "images" / split).glob("*.jpg")))
    print(f"[{split}] 이미지 {n_images}장")

## 2. 학습: YOLOv8n

하이퍼파라미터는 `train_yolo_tile_augmented.ipynb`의 YOLOv8s/YOLO26n 학습 셀과 동일하다
(`epochs=100`, `imgsz=960`, `batch=-1`, `patience=25`, `seed=42`, `deterministic=True`,
`close_mosaic=10`). `batch=-1`(자동 배치)은 모델 크기에 따라 실제 배치 크기가 달라질 수
있다는 점에 유의 -- `EXPERIMENT_PLAN.md` 4.2절 참고. 셋 다 자동 배치를 그대로 쓰기로
했다면 이 값 그대로 두면 된다.

In [ ]:
from ultralytics import YOLO

YOLO_EPOCHS = 100
YOLO_IMGSZ = 960

yolov8n_model = YOLO("yolov8n.pt")
yolov8n_model.train(
    data=str(dataset_yaml_path.resolve()),
    epochs=YOLO_EPOCHS,
    imgsz=YOLO_IMGSZ,
    batch=-1,
    device=0 if torch.cuda.is_available() else "cpu",
    workers=8,
    patience=25,
    project="runs/detect",
    name="rummikub_yolov8n_tile_augmented",
    pretrained=True,
    seed=RANDOM_SEED,
    deterministic=True,
    close_mosaic=10,
    plots=True,
)

yolov8n_best_path = Path(yolov8n_model.trainer.best)
yolov8n_out_path = WEIGHTS_OUT / "yolov8n_tile_detector_best.pt"
shutil.copy2(yolov8n_best_path, yolov8n_out_path)
print("YOLOv8n 가중치 저장:", yolov8n_out_path.resolve())

## 3. test 성능 3-way 비교

방금 학습한 YOLOv8n과, 이미 `weights/`에 있는 YOLOv8s/YOLO26n 가중치를 같은 test split으로
평가해서 나란히 비교한다 (재학습하지 않고 저장된 가중치를 로드만 함). 두 가중치 파일이 이
노트북을 돌리는 머신의 `weights/`에 없으면 해당 모델 비교는 건너뛴다.

In [ ]:
def eval_on_test(model: YOLO, name: str):
    metrics = model.val(
        data=str(dataset_yaml_path.resolve()),
        split="test",
        imgsz=YOLO_IMGSZ,
        device=0 if torch.cuda.is_available() else "cpu",
        project="runs/detect",
        name=f"{name}_eval",
        plots=True,
    )
    print(
        f"{name:12s} P={metrics.box.mp:.3f} R={metrics.box.mr:.3f} "
        f"mAP50={metrics.box.map50:.3f} mAP50-95={metrics.box.map:.3f}"
    )


eval_on_test(yolov8n_model, "yolov8n")

for name, weight_name in (("yolov8s", "yolov8s_tile_detector_best.pt"), ("yolo26n", "yolo26n_tile_detector_best.pt")):
    weight_path = WEIGHTS_OUT / weight_name
    if not weight_path.is_file():
        print(f"[skip] {weight_path.resolve()}가 없어 {name} 비교는 건너뜁니다.")
        continue
    eval_on_test(YOLO(str(weight_path)), name)

## 참고

- `weights/yolov8n_tile_detector_best.pt`가 만들어지면, `EXPERIMENT_PLAN.md` 7.2절의
  선행 구현(`pipeline_jt.py`에 YOLOv8n variant 추가)을 진행할 수 있다.
- 이 노트북은 학습/평가만 담당한다 -- Jetson 실기 latency 비교(실험 4)와 holdout 카메라
  이미지 정확도 비교(실험 1)는 별도로 진행한다.